# 03. Data Cleaning
In the Employees notebooks, missing values were filled with the **mean** (`Age`) or the **mode** (`DistanceFromHome`). For prices we check the data in the same way — but we will **not** fill anything, and we will see why.

* **Needs:** the four price files. **Makes:** `work/clean_ISEQ.csv`, `work/clean_GSPC.csv`, `work/clean_GDAXI.csv`, `work/clean_FTSE.csv`.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

## Data Reading (with the dates already as the index)
`index_col=0` uses the first column as the index and `parse_dates=True` turns it into dates — the work of notebook 02 in one line.

In [2]:
from pandas import read_csv
data1 = read_csv(folder + 'ISEQ.csv', index_col=0, parse_dates=True)
data2 = read_csv(folder + 'GSPC.csv', index_col=0, parse_dates=True)
data3 = read_csv(folder + 'GDAXI.csv', index_col=0, parse_dates=True)
data4 = read_csv(folder + 'FTSE.csv', index_col=0, parse_dates=True)
data1.head()

,Open,High,Low,Close,Volume
2002-10-01,3838.520020,3882.870117,3813.500000,3863.100098,0
2002-10-02,3864.139893,3942.629883,3859.600098,3926.350098,0
2002-10-03,3926.270020,3933.560059,3871.360107,3875.959961,0
2002-10-04,3875.870117,3887.030029,3840.580078,3842.899902,0
2002-10-07,3843.040039,3843.040039,3736.889893,3748.659912,0


## Missing values

In [3]:
data1.isna().sum()

Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

In [4]:
print(data2.isna().sum().sum())
print(data3.isna().sum().sum())
print(data4.isna().sum().sum())

0
0
0


No missing values. **If** there were, we would **remove** the row instead of filling it: a missing price means the market was closed, and an invented price (for example the mean) would create a fake movement.

In [5]:
data1 = data1.dropna(subset=['Close'])
data2 = data2.dropna(subset=['Close'])
data3 = data3.dropna(subset=['Close'])
data4 = data4.dropna(subset=['Close'])
print(data1.shape, data2.shape, data3.shape, data4.shape)

(5898, 5) (5850, 5) (5903, 5) (5873, 5)


## Impossible values
A price can never be zero or negative.

In [6]:
print((data1['Close'] <= 0).sum())
print((data2['Close'] <= 0).sum())
print((data3['Close'] <= 0).sum())
print((data4['Close'] <= 0).sum())

0
0
0
0


In [7]:
data1 = data1[data1['Close'] > 0]
data2 = data2[data2['Close'] > 0]
data3 = data3[data3['Close'] > 0]
data4 = data4[data4['Close'] > 0]

## Duplicated dates
Each date should appear only once.

In [8]:
print(data1.index.duplicated().sum())
print(data2.index.duplicated().sum())
print(data3.index.duplicated().sum())
print(data4.index.duplicated().sum())

0
0
0
0


## A hidden problem: 'stale' opening prices
There is a better way to measure daily volatility that uses the **opening** price (the Garman–Klass method). But on Yahoo, the ISEQ's opening price is often just **yesterday's closing price copied** — not a real price. Let us count how often.

`shift(1)` moves a column down by one row, so each row sees **yesterday's** close:

In [9]:
data1['previous_close'] = data1['Close'].shift(1)
data1[['Open', 'Close', 'previous_close']].loc['2015-03-02':'2015-03-06']

,Open,Close,previous_close
2015-03-02,5977.029785,5957.620117,5977.029785
2015-03-03,5957.620117,5893.169922,5957.620117
2015-03-04,5893.169922,5954.250000,5893.169922
2015-03-05,5927.680176,6017.759766,5954.250000
2015-03-06,6017.759766,6021.169922,6017.759766


In this week, on four of the five days the `Open` is exactly the `previous_close`. Let us mark these days with `True`:

In [10]:
data1['stale_open'] = (data1['Open'] - data1['previous_close']).abs() < 0.000000001
print('2003-2006:', round(data1.loc['2003':'2006', 'stale_open'].mean() * 100, 1), '%')
print('2007-2012:', round(data1.loc['2007':'2012', 'stale_open'].mean() * 100, 1), '%')
print('2013-2019:', round(data1.loc['2013':'2019', 'stale_open'].mean() * 100, 1), '%')
print('2020-2025:', round(data1.loc['2020':'2025', 'stale_open'].mean() * 100, 1), '%')

2003-2006: 8.3 %
2007-2012: 37.7 %
2013-2019: 75.4 %
2020-2025: 10.0 %


(The mean of `True`/`False` is the share of `True`.) In 2013–2019, three days out of four have a copied opening price. A volatility measure built on these would be wrong. **Decision: we use only closing prices.**

## Save the clean data
We keep the closing price (and, for the ISEQ, the high and low prices, used in a later robustness check). The files go into a sub-folder `work`.

In [11]:
import os
os.makedirs(folder + 'work', exist_ok=True)

data1[['High', 'Low', 'Close']].to_csv(folder + 'work/clean_ISEQ.csv')
data2[['Close']].to_csv(folder + 'work/clean_GSPC.csv')
data3[['Close']].to_csv(folder + 'work/clean_GDAXI.csv')
data4[['Close']].to_csv(folder + 'work/clean_FTSE.csv')
print('saved')

saved
